### Установка библиотек для Colab

In [ ]:
# !pip install docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.4/451.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.3/269.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 112.3 MB/s eta 0:00:00
   ━━━

In [ ]:
# !pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 89.9 MB/s eta 0:00:00


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


### 0. Импорты

In [ ]:
import sqlite3

import pandas as pd

# Отобразить все строки
pd.set_option("display.max_rows", None)

In [4]:
import re
import fitz  # PyMuPDF
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling_core.types.doc import PictureItem, TableItem, TextItem, DocItemLabel

### 1. Подключение к БД

In [ ]:
# Путь к файлу базы данных
db_path = "../data/db/construction.db"

# Устанавливаем соединение
conn = sqlite3.connect(db_path)

In [ ]:
# 1. Посмотреть список всех таблиц
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Таблицы в базе:")
print(tables)

Таблицы в базе:
              name
0          objects
1  sqlite_sequence
2            works
3         progress
4      contractors


In [ ]:
# 2. Сохраним таблицу works в DataFrame
df = pd.read_sql("SELECT * FROM works", conn)
conn.close()

In [ ]:
df.to_csv("../data/docs/db_works.csv")

| Вид работ               | Частота |
| ----------------------- | ------- |
| Вентиляция              | 30      |
| Электроснабжение        | 18      |
| Обустройство территории | 18      |
| Окраска                 | 18      |
| Монтаж перекрытий       | 18      |
| Установка дверей        | 17      |
| Внешняя отделка         | 16      |
| Внутренняя отделка      | 12      |
| Земляные работы         | 13      |

Начнем со следующих СП:
* СП 60.13330 (СНиП 41-01-2003) «Отопление, вентиляция и кондиционирование воздуха».
* СП 70.13330 (СНиП 3.03.01-87) «Несущие и ограждающие конструкции».
* СП 45.13330 (СНиП 3.02.01-87) «Земляные сооружения, основания и фундаменты».

### 2. Настройка парсера документа

#### 2.1 Автоматизированная очистка и экстракция (Docling)

In [6]:
# Настройки путей
# первый документ
doc_path = "/content/drive/MyDrive/Colab_Notebooks/rag_docs/data/raw/document_sp_60.pdf"
output_dir = Path("/content/drive/MyDrive/Colab_Notebooks/rag_docs/data/extracted/images_sp_60_v3")
output_dir.mkdir(parents=True, exist_ok=True)
output_md_path = "/content/drive/MyDrive/Colab_Notebooks/rag_docs/data/extracted/sp60_fcc_cleaned_v3.md"

In [7]:
# 1. Настройка пайплайна под специфику СП (СНиП)
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
pipeline_options.generate_picture_images = True
pipeline_options.generate_table_images = True
pipeline_options.do_ocr = True
pipeline_options.do_formula_enrichment = True   # Включает распознавание формул

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

In [8]:
# Функция для эвристического поиска формул (если Docling не распознал)
def is_likely_formula(text: str) -> bool:
    """Проверяет, похож ли текст на формулу по наличию математических символов."""
    math_symbols = re.compile(r'[=<>+\-*/^_{}\[\]()∑∏∫√∂∞≈≠≤≥]')
    if math_symbols.search(text):
        return True
    if re.search(r'\d+\s*[=<>]\s*\d+', text):
        return True
    return False

In [9]:
# 2. Основной цикл обработки
result = converter.convert(doc_path)
pdf_doc = fitz.open(doc_path)
doc_md = result.document.export_to_markdown()

HEADER_LIMIT = 80
FOOTER_LIMIT = 760

image_counter = 0
table_counter = 0
formula_counter = 0
image_replacements = []
formula_replacements = []

print("Начинаю экспорт изображений и формул...")

for item, _ in result.document.iterate_items():
    # --- ОБЫЧНЫЕ ИЗОБРАЖЕНИЯ ---
    if isinstance(item, PictureItem):
        bbox = item.prov[0].bbox
        if bbox.t < HEADER_LIMIT or bbox.b > FOOTER_LIMIT:
            image_replacements.append("")
            continue
        if item.image and item.image.pil_image:
            image_counter += 1
            filename = f"img_p{item.prov[0].page_no}_{image_counter}.png"
            item.image.pil_image.save(output_dir / filename)
            image_replacements.append(f"![Изображение](images/{filename})")
            print(f"Сохранено изображение: {filename}")
        continue

    # --- ТАБЛИЦЫ ---
    if isinstance(item, TableItem) and item.image and item.image.pil_image:
        table_counter += 1
        filename = f"tab_p{item.prov[0].page_no}_{table_counter}.png"
        item.image.pil_image.save(output_dir / filename)

    # --- ФОРМУЛЫ ---
    is_formula = False
    formula_latex = None

    if hasattr(item, 'label') and item.label == DocItemLabel.FORMULA:
        bbox = item.prov[0].bbox
        page_num = item.prov[0].page_no
        formula_counter += 1

        # Пытаемся получить LaTeX-код
        formula_latex = None
        if hasattr(item, 'orig_text') and item.orig_text:
            formula_latex = item.orig_text.strip()
        elif hasattr(item, 'text') and item.text:
            formula_latex = item.text.strip()

        # Проверяем качество LaTeX: если содержит кириллицу или слишком короткий/длинный, лучше вырезать картинку
        bad_latex = False
        if formula_latex:
            # Если в LaTeX больше 2 слов на кириллице (русские буквы), скорее всего это не формула
            cyrillic_words = len(re.findall(r'[а-яА-ЯёЁ]{2,}', formula_latex))
            if cyrillic_words > 2:
                bad_latex = True
            # Если строка слишком длинная (>200 символов) и нет типичных мат. символов, тоже не формула
            if len(formula_latex) > 200 and not re.search(r'[=<>+\-*/^_{}\[\]()∑∏∫√∂∞≈≠≤≥]', formula_latex):
                bad_latex = True
        else:
            bad_latex = True  # Нет LaTeX — вырезаем картинку

        if formula_latex and not bad_latex:
            # Вставляем LaTeX
            md_formula = f"$$\n{formula_latex}\n$$"
            original_text = getattr(item, 'orig_text', None) or getattr(item, 'text', '')
            if original_text and original_text in doc_md:
                doc_md = doc_md.replace(original_text, md_formula)
            else:
                doc_md += f"\n\n{md_formula}\n"
            print(f"Распознана формула (LaTeX) на стр. {page_num}: {formula_latex[:60]}...")
        else:
            # Вырезаем как изображение с нормализацией координат
            left = min(bbox.l, bbox.r)
            right = max(bbox.l, bbox.r)
            top = min(bbox.t, bbox.b)
            bottom = max(bbox.t, bbox.b)

            width = right - left
            height = bottom - top

            if width < 5 or height < 5:
                print(f"Пропущена формула на стр. {page_num}: слишком маленькая область ({width:.1f}x{height:.1f})")
                continue

            rect = fitz.Rect(left, top, right, bottom)
            pdf_page = pdf_doc[page_num - 1]

            page_rect = pdf_page.rect
            if rect.x0 < page_rect.x0 or rect.y0 < page_rect.y0 or \
               rect.x1 > page_rect.x1 or rect.y1 > page_rect.y1:
                print(f"Пропущена формула на стр. {page_num}: bbox выходит за границы страницы")
                continue

            matrix = fitz.Matrix(3.0, 3.0)
            pix = pdf_page.get_pixmap(matrix=matrix, clip=rect)

            if pix is None or pix.w <= 0 or pix.h <= 0:
                print(f"Не удалось создать изображение формулы на стр. {page_num}")
                continue

            filename = f"sp60_formula_p{page_num}_{formula_counter}.png"
            save_path = output_dir / filename
            pix.save(save_path)

            md_link = f"![Формула](images/{filename})"
            original_text = getattr(item, 'text', '')
            if original_text and original_text in doc_md:
                doc_md = doc_md.replace(original_text, md_link)
            else:
                formula_replacements.append(md_link)
            print(f"Вырезана формула как изображение: {filename} (стр. {page_num})")

pdf_doc.close()

[INFO] 2026-04-07 13:45:44,673 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-07 13:45:44,677 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-07 13:45:44,680 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.7.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-07 13:45:45,470 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-04-07 13:45:45,649 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-07 13:45:45,651 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-07 13:45:46,179 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-07 13:45:46,180 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-07 13:45:46,182 [RapidOCR] download_file.py:68: Initiat

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Passing `generation_config` together with generation-related arguments=({'use_cache', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Начинаю экспорт изображений и формул...
Распознана формула (LaTeX) на стр. 59: l _ { z } = 4 D \frac { q } { q _ { z } } \geq 1 0 ,...
Распознана формула (LaTeX) на стр. 80: G _ { \max } = \Pi \Pi H \Psi \cdot L _ { o 0 u } ,...
Распознана формула (LaTeX) на стр. 113: Q _ { o b } ^ { p } = \Sigma _ { n } \left ( Q _ { t p _ { n...
Распознана формула (LaTeX) на стр. 113: Q _ { _ { T P _ { n } } } = \left ( t _ { _ { B _ { n } } } ...
Распознана формула (LaTeX) на стр. 113: Q _ { \mathbb { I } _ { n } } = & \left ( t _ { _ { B _ { n ...
Распознана формула (LaTeX) на стр. 114: K _ { i } = \frac { 1 } { R _ { 0 , i } ^ { \text {TP} } } \...
Распознана формула (LaTeX) на стр. 115: Q _ { _ { \text {BETHT} _ { n } } } = \left ( t _ { _ { \tex...
Распознана формула (LaTeX) на стр. 115: \rho _ { _ { H } } = \frac { 3 5 3 } { 2 7 3 + t _ { _ { H }...
Распознана формула (LaTeX) на стр. 115: Q _ { \i h \phi _ { n } } = ( t _ { _ { B } n } - t _ { _ { ...
Распознана формула (LaTeX) на стр. 116: G _

#### 2.2 Нормализация текста (Cleaning Script)

In [ ]:
# Замена оставшихся тегов-заглушек
def replace_image_tag(match):
    return image_replacements.pop(0) if image_replacements else match.group(0)

def replace_formula_tag(match):
    return formula_replacements.pop(0) if formula_replacements else match.group(0)

doc_md = re.sub(r'<!--\s*image\s*-->', replace_image_tag, doc_md)
doc_md = re.sub(r'<!--\s*formula-not-decoded\s*-->', replace_formula_tag, doc_md)

# Очистка от колонтитулов и шума
doc_md = re.sub(r"(?m)^[\s\*\#]*СП\s*60\.13330\.2020[\s\*\#]*$\n?", "", doc_md)
doc_md = re.sub(r"(?m)^\s*([IVXLCDM]+|\d+)\s*$\n", "", doc_md)
doc_md = re.sub(r"(## Приложение [А-Я])\n+##\s*(.*)", r"\1: \2", doc_md)
doc_md = re.sub(r"(?i)т\s+а\s+б\s+л\s+и\s+ц\s+а", "Таблица", doc_md)
doc_md = re.sub(r"(?i)п\s*р\s*и\s*м\s*е\s*ч\s*а\s*н\s*и\s*[яе]", "Примечания", doc_md)
doc_md = re.sub(r"#{1,6}\s*(Примечание.*)", r"**\1**", doc_md)

# Сохранение результата
with open(output_md_path, "w", encoding="utf-8") as f:
    f.write(doc_md)

print(f"\nГотово! Сохранено: изображений {image_counter}, таблиц {table_counter}, формул {formula_counter}")
print(f"Результат в файле: {output_md_path}")


Готово! Сохранено: изображений 2, таблиц 26, формул 56
Результат в файле: /content/drive/MyDrive/Colab_Notebooks/rag_docs/data/extracted/sp60_fcc_cleaned_v3.md
